# PlantCLEF 2015 Local Crop Diagnostics

Checks whether the current S-CNN(B) local `32x32` center crop on official LeafScan test images contains leaf foreground or mostly background.

## 1. Clone Or Update Project

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_DIR = Path('/content/diploma')
REPO_URL = 'https://github.com/robodanill/diploma.git'
BRANCH = 'robodanill/main'


def clone_project():
    os.chdir('/content')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)


def pull_project() -> bool:
    if not (PROJECT_DIR / '.git').exists():
        return False
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
    return result.returncode == 0


if PROJECT_DIR.exists():
    print(f'Trying to update existing project: {PROJECT_DIR}')
    if not pull_project():
        print('Pull failed or project is not a git repository; cloning a fresh copy.')
        clone_project()
else:
    print(f'Project not found at {PROJECT_DIR}; cloning a fresh copy.')
    clone_project()

os.chdir(PROJECT_DIR)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[ml]'], check=True)

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print(f'Project commit: {commit}')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Restore Official LeafScan Test Data

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

TEST_ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz
if [ ! -f "$TEST_ARCHIVE" ]; then
  echo "Missing test archive: $TEST_ARCHIVE" >&2
  exit 2
fi

rm -rf data/plantclef2015/test_leafscan
mkdir -p data/plantclef2015/test_leafscan
tar -xzf "$TEST_ARCHIVE" -C data/plantclef2015/test_leafscan
test -f data/plantclef2015/test_leafscan/leafscan/metadata.csv
cp data/plantclef2015/test_leafscan/leafscan/metadata.csv data/plantclef2015/test_leafscan_metadata.csv

python - <<'PY'
import csv
from pathlib import Path
metadata = Path('data/plantclef2015/test_leafscan_metadata.csv')
with metadata.open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
print('test rows:', len(rows))
print('test genera:', len({row['genus'] for row in rows}))
print('test species:', len({row['species'] for row in rows}))
PY

## 4. Compute Crop Foreground Statistics

In [ ]:
from pathlib import Path
import csv
import random

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

from plant_classifier.preprocessing.views import LeafBoundingBoxCrop, _leaf_mask

PROJECT_DIR = Path('/content/diploma')
TEST_ROOT = PROJECT_DIR / 'data/plantclef2015/test_leafscan/leafscan'
TEST_METADATA = PROJECT_DIR / 'data/plantclef2015/test_leafscan_metadata.csv'
DRIVE_OUT = Path('/content/drive/MyDrive/diploma_diagnostics/local_crop')
OUT_DIR = DRIVE_OUT if DRIVE_OUT.parent.exists() else PROJECT_DIR / 'diagnostics/local_crop'
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
CROP_SIZE = 32
MORPHOLOGY_SIZE = 5
BBOX_PADDING = 4
RESAMPLE_BILINEAR = Image.Resampling.BILINEAR if hasattr(Image, 'Resampling') else Image.BILINEAR


def resolve_image_path(row: dict) -> Path:
    path = Path(row['image_path'])
    return path if path.is_absolute() else TEST_ROOT / path


def center_crop_box(width: int, height: int, crop_size: int) -> tuple[int, int, int, int]:
    left = (width - crop_size) // 2
    top = (height - crop_size) // 2
    return left, top, left + crop_size, top + crop_size


cropper = LeafBoundingBoxCrop(padding=BBOX_PADDING, morphology_size=MORPHOLOGY_SIZE)
crop_box = center_crop_box(IMAGE_SIZE, IMAGE_SIZE, CROP_SIZE)

with TEST_METADATA.open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))

records = []
for row in rows:
    image_path = resolve_image_path(row)
    with Image.open(image_path) as image:
        rgb = image.convert('RGB')
    bbox_image = cropper(rgb)
    resized = bbox_image.resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE_BILINEAR)
    local_crop = resized.crop(crop_box)
    resized_mask = _leaf_mask(resized, MORPHOLOGY_SIZE)
    crop_mask = resized_mask.crop(crop_box)

    crop_mask_array = np.asarray(crop_mask)
    crop_array = np.asarray(local_crop)
    foreground_fraction = float((crop_mask_array > 0).mean())
    near_white_fraction = float((crop_array.min(axis=2) >= 245).mean())
    records.append(
        {
            'image_path': str(image_path),
            'image_name': image_path.name,
            'family': row['family'],
            'genus': row['genus'],
            'species': row['species'],
            'original_width': rgb.width,
            'original_height': rgb.height,
            'bbox_width': bbox_image.width,
            'bbox_height': bbox_image.height,
            'bbox_area_ratio': (bbox_image.width * bbox_image.height) / max(1, rgb.width * rgb.height),
            'local_foreground_fraction': foreground_fraction,
            'local_background_fraction': 1.0 - foreground_fraction,
            'local_near_white_fraction': near_white_fraction,
            'local_rgb_mean': float(crop_array.mean()),
            'local_rgb_std': float(crop_array.std()),
        }
    )

diagnostics = pd.DataFrame(records)
csv_path = OUT_DIR / 'test_local_crop_diagnostics.csv'
diagnostics.to_csv(csv_path, index=False)

print('Saved:', csv_path)
print('rows:', len(diagnostics))
display(diagnostics[['local_foreground_fraction', 'local_near_white_fraction', 'bbox_area_ratio']].describe())
for threshold in [0.05, 0.10, 0.25, 0.50, 0.75]:
    count = int((diagnostics['local_foreground_fraction'] <= threshold).sum())
    print(f'foreground <= {threshold:.2f}: {count}/{len(diagnostics)}')

display(
    diagnostics.sort_values('local_foreground_fraction')
    [['image_name', 'genus', 'species', 'local_foreground_fraction', 'local_near_white_fraction', 'bbox_area_ratio']]
    .head(20)
)

## 5. Plot Distribution

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(diagnostics['local_foreground_fraction'], bins=24, color='#0f766e', edgecolor='white')
axes[0].set_xlabel('foreground fraction in 32x32 crop')
axes[0].set_ylabel('test image count')
axes[0].set_title('Leaf foreground in local crop')
axes[0].grid(True, alpha=0.25)

axes[1].scatter(
    diagnostics['local_foreground_fraction'],
    diagnostics['local_near_white_fraction'],
    s=18,
    alpha=0.75,
    color='#334155',
)
axes[1].set_xlabel('foreground fraction')
axes[1].set_ylabel('near-white pixel fraction')
axes[1].set_title('Crop foreground vs. white background')
axes[1].grid(True, alpha=0.25)

fig.tight_layout()
hist_path = OUT_DIR / 'test_local_crop_foreground_distribution.png'
fig.savefig(hist_path, dpi=180)
print('Saved:', hist_path)
plt.show()

## 6. Visual Audit: Random And Worst Crops

In [ ]:
from PIL import ImageDraw

RESAMPLE_NEAREST = Image.Resampling.NEAREST if hasattr(Image, 'Resampling') else Image.NEAREST


def load_crop_views(image_path: Path):
    with Image.open(image_path) as image:
        original = image.convert('RGB')
    bbox_image = cropper(original)
    resized = bbox_image.resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE_BILINEAR)
    boxed = resized.copy()
    draw = ImageDraw.Draw(boxed)
    draw.rectangle(crop_box, outline=(220, 38, 38), width=3)
    crop = resized.crop(crop_box).resize((160, 160), RESAMPLE_NEAREST)
    mask = _leaf_mask(resized, MORPHOLOGY_SIZE).crop(crop_box).convert('RGB').resize((160, 160), RESAMPLE_NEAREST)
    return original, boxed, crop, mask


def make_grid(frame: pd.DataFrame, output_path: Path, title: str) -> None:
    frame = frame.reset_index(drop=True)
    fig, axes = plt.subplots(len(frame), 4, figsize=(11, max(2.2, 2.0 * len(frame))))
    if len(frame) == 1:
        axes = np.array([axes])
    headers = ['original', 'bbox resized + crop box', '32x32 crop', 'crop mask']
    for column, header in enumerate(headers):
        axes[0, column].set_title(header, fontsize=10)
    for row_index, item in frame.iterrows():
        views = load_crop_views(Path(item['image_path']))
        label = f"{item['genus']} | fg={item['local_foreground_fraction']:.2f} | {item['image_name']}"
        for column, view in enumerate(views):
            axes[row_index, column].imshow(view)
            axes[row_index, column].axis('off')
        axes[row_index, 0].set_ylabel(label, fontsize=8)
    fig.suptitle(title, y=1.0, fontsize=12)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    print('Saved:', output_path)
    plt.show()


random_examples = diagnostics.sample(min(8, len(diagnostics)), random_state=42)
worst_examples = diagnostics.sort_values('local_foreground_fraction').head(min(8, len(diagnostics)))

make_grid(random_examples, OUT_DIR / 'test_local_crop_random_examples.png', 'Random test local crops')
make_grid(worst_examples, OUT_DIR / 'test_local_crop_low_foreground_examples.png', 'Lowest-foreground test local crops')

## 7. Why The 32x32 Crop Is Resized To 224x224

The current S-CNN(B) backbone is VGG16 initialized from ImageNet weights, so the code follows the ImageNet/VGG convention and feeds a 224x224 tensor. A raw 32x32 tensor is not what the pretrained VGG16 filters were trained on; after the VGG pooling stack, most spatial detail would collapse. Keeping 32x32 natively would require a deliberately modified backbone/head or a smaller CNN. Resizing keeps the pretrained VGG16 pipeline usable, but it also means the model sees an enlarged patch, not a native 32x32 pipeline.